# 章节练习

本练习覆盖**图编译**模块（3.2–3.6）的核心知识点：AscendIR 构图与 Parser 解析、编译配置（融合/精度/buffer/stream）、编译产物（OM 结构/外置权重/SO in OM/模型缓存）、自定义算子入图、以及自定义融合 Pass。

题目按四层组织——判断题、单选题、多选题、实践题。前三层为基础自测，实践题分为「跑通闭环」与「改动实验」两类，建议结合 GE 仓样例（`ge/examples/`）动手完成。如有错误，建议回到对应小节复盘，温故而知新。

| 层级 | 练习类型 | 验收方式 |
| --- | --- | --- |
| 基础自测 | 判断 / 单选 / 多选题 | 对照 answer 参考答案 |
| 跑通闭环 | 按课程步骤完成一次完整操作 | Notebook/终端输出产物与关键日志 |
| 改动实验 | 修改配置/输入观察行为变化 | 记录现象并说明定位入口 |

## 一、判断题

1. （判断题）GE 接受两类图输入：用 AscendIR API 手工构图，或通过 Parser（如 `aclgrphParseONNX`）解析 ONNX/TensorFlow 等框架模型。

2. （判断题）编译配置只有一个全局层级，不存在会话级与图级的区分。

3. （判断题）开启外置权重后，权重数据从 OM 内部移出、独立存放，可在多模型/多实例间共享，有利于部署。

4. （判断题）自定义算子入图时，`REG_OP` 的 op type 名、`REG_AUTO_MAPPING_OP` 的类名、构图侧使用的 op type 可以各不相同，GE 会自动模糊匹配。

5. （判断题）基于 Pattern 匹配实现的自定义融合 Pass 支持在 Pattern 中匹配控制边和嵌套子图。

6. （判断题）自定义算子和自定义融合 Pass 都不需要修改 GE 框架源码，分别以 `.so` 交付件和 `.so` 插件的形式被 GE 加载。

7. （判断题）模型编译缓存命中后可以跳过重复编译，但当 soc_version、编译选项或图结构发生变化时缓存会失效。

## 二、单选题

8. （单选题）下列关于「手工构图」与「Parser 导入」的选择建议，哪个最合适？
    A. 任何场景都应优先手工逐算子构图
    B. 已有 ONNX/pb 模型迁移时优先用 Parser 解析，全新构图或需精细控制时用 AscendIR API
    C. Parser 只能解析 ONNX，不能解析 TensorFlow
    D. 两种方式产出的是不同的 IR，无法进入同一编译流程

9. （单选题）`precision_mode` 这类精度配置主要用于解决什么问题？
    A. 控制 OM 文件的存放路径
    B. 在精度与性能之间做取舍（如是否允许降精度以提升性能）
    C. 决定使用哪种 Parser
    D. 指定自定义算子的注册名

10. （单选题）「SO in OM」机制的作用是？
    A. 把 ONNX 模型打包进 OM
    B. 把自定义算子的 `.so` 随模型一起打包发布，便于部署
    C. 关闭所有融合 Pass
    D. 把权重外置到独立文件

11. （单选题）V2（BaseCustomOp）机制中，运行时负责获取输入、分配输出、launch kernel 的核心接口是？
    A. `ShapeInferOp::InferShape`
    B. `EagerExecuteOp::Execute`
    C. `PortableOp::Serialize`
    D. `ArgsUpdater::UpdateHostArgs`

12. （单选题）`PatternFusionPass` 中**必须**重写、用于定义匹配模板的函数是？
    A. `MeetRequirements`
    B. `Patterns`
    C. `Run`
    D. `Compile`

13. （单选题）自定义融合 Pass 初次开发时，最常用、且 replacement 后续会进入统一 shape 推导流程的执行阶段是？
    A. `kAfterInferShape`
    B. `kBeforeInferShape`
    C. `kAfterBuiltinFusionPass`
    D. `kAfterOriginGraphOptimize`

14. （单选题）在支持 GE V2 的版本中，要把自定义算子 binary 下沉到离线 OM，相比纯在线执行通常额外需要哪一组接口？
    A. 只需 `ShapeInferOp`
    B. `CompilableOp` + `PortableOp`
    C. 只需 `ArgsUpdater`
    D. 不需要任何额外接口

## 三、多选题

15. （多选题）以下关于编译配置的描述，哪些正确？
    A. 配置体系通常分全局选项、Session 选项、图级选项三层，存在优先级关系
    B. 融合配置可以开关融合规则、设置融合白/黑名单，用于调试时关闭指定融合 Pass
    C. 多 stream 配置可用于多 stream 并行，但 stream 数量有上限
    D. buffer / 内存复用配置一旦开启就无法关闭，且作用范围不可控

16. （多选题）以下关于自定义算子入图的描述，哪些正确？
    A. 自定义算子代码在 GE 仓外维护，通过 `ASCEND_CUSTOM_OPP_PATH` 加载，无需改 GE 框架
    B. V2 能力接口（Execute/InferShape/Compile/Serialize 等）可按需组合，GE 通过 dynamic_cast 检测
    C. TensorFlow 场景 `REG_AUTO_MAPPING_OP` 可从 TF 原型自动生成 GE proto，无需额外编写 `REG_OP`
    D. fallback 下发适用于离线推理场景

17. （多选题）以下关于自定义融合 Pass 的描述，哪些正确？
    A. 有两条实现路线：基于 Pattern 匹配（推荐）与基于改图接口
    B. Pattern 边界规则要求外部输入和「替换后外部仍使用的输出」都要声明
    C. `DecomposePass` 适合把单个算子展开为多个算子
    D. Pass `.so` 安装后需要重新编译 GE 才能生效

18. （多选题）以下关于验证手段的描述，哪些正确？
    A. `DUMP_GE_GRAPH=1` 可 dump 编译过程中的图，用于对比融合前后结构
    B. 自定义算子离线链路成功时可观察 ATC 生成 `.om` 文件、ACL 执行打印成功信息
    C. 并发是否真有收益需结合 Profiling 判断，仅看 dump 图无法确认性能收益
    D. 只要图结构看起来对，就一定有性能提升，无需 Profiling

## 四、实践题

实践题分「跑通闭环」与「改动实验」两类，建议在已安装 CANN 9.0.0（具备 `atc`、GE/ACL 头文件与库，并 `source set_env.sh`）的 Ascend910B4 环境下完成。3.05 的在线与离线源码位于本章 `Sources/03.05/` 下。

### （一）跑通闭环

**实践题 19｜自定义算子离线 OM 闭环（综合 3.2 / 3.4 / 3.5；可运行离线样例见 3.5）**

目标：跑通 `Ascend C 自定义算子 → 构图导出 AIR → ATC 离线编译 OM → ACL 执行` 全链路。

步骤（参考 `Sources/03.05/offline_add_custom/`）：

1. 进入样例目录，执行 `SOC_VERSION=Ascend910B4 bash run.sh`，脚本会依次完成以下五步：
   - **Step 1**：cmake 配置并编译算子工程
   - **Step 2**：编译 Ascend C kernel 包（`cmake --build --target binary`）
   - **Step 3**：安装 OPP 包并设置 `ASCEND_CUSTOM_OPP_PATH`，运行 `bin/offline_add_graph_build` 导出 `output/single_add.air`
   - **Step 4**：用 `atc --model=single_add.air --framework=1 --output=single_add --soc_version=Ascend910B4` 将 AIR 编译为 `output/single_add.om`
   - **Step 5**：运行 `bin/offline_add_model_exec` 加载并执行 `output/single_add.om`

2. 如需分步执行，可手动运行 `cmake -S . -B build -DCMAKE_BUILD_TYPE=Release` → `cmake --build build -j$(nproc)` → `cmake --build build --target binary` → `cmake --install build --prefix build/install`，然后设置 `ASCEND_CUSTOM_OPP_PATH` 指向 `build/install/vendors/customize`（须包含 `op_proto`、`op_impl` 子目录），再执行 `build/bin/offline_add_graph_build` 和 `build/bin/offline_add_model_exec`。

验收：`output/` 下生成 `single_add.air` 与 `single_add.om`；终端打印 `First element of output: 3.000000` 和 `[OK] AIR -> ATC -> OM -> ACL offline validation`。

**实践题 20｜自定义融合 Pass 闭环 + 验证（对应 3.6）**

目标：跑通 `MatMul+Add → GEMM` 自定义融合 Pass，并验证生效。

步骤（参考 `ge/examples/fusion_pass/pattern_base_pass/1_fuse_matmul_add_pass/cpp/`）：
1. `source set_env.sh`，`mkdir build && cd build && cmake .. && make -j$(nproc) && make install`（安装 `.so` 到 `opp/vendors/<xxx>/custom_fusion_passes/`）。
2. `export DUMP_GE_GRAPH=1`，导出 onnx 后用 ATC 编译，或执行在线推理脚本。
3. 对比 `ge_onnx_xxxx_PreRunBegin.pbtxt`（融合前）与 `ge_onnx_xxxx_RunCustomPassBeforeInfershape.pbtxt`（融合后）。

验收：日志出现 `Define pattern for FuseMatMulAndAddPass` / `Define replacement ...`；dump 图中 `MatMul + Add` 已被单个 `GEMM` 替换（用 Netron 查看）。

### （二）改动实验

**实践题 21｜编译配置改动观察（对应 3.3）**

在跑通的构图/编译流程上，尝试修改一项编译配置并观察变化：
- 关闭某个内置融合 Pass（融合黑名单），对比 dump 图中算子是否不再融合；
- 或调整 `precision_mode`，观察是否影响算子精度模式选择；
- 或调整 stream 配置，结合 Profiling 观察并行度变化。

验收要点：用一句话说明你改了哪一层配置（全局/会话/图级）、预期现象、实际现象，以及定位入口（dump 图 / 日志 / Profiling）。

**实践题 22｜自定义算子三处命名一致性实验（对应 3.5）**

故意把 `REG_AUTO_MAPPING_OP` 的类名或构图侧 op type 改成与 `REG_OP` 不一致，重新编译运行，观察报错。

验收要点：复现「GE 找不到自定义算子」类错误，并说明正确做法是保证 REG_OP 名 / 注册类名 / 构图侧 op type **三者一致**。CANN 9.0 在线场景下，还需检查 `ASCEND_CUSTOM_OPP_PATH` 的某个路径项是否直接指向包含 `libcust_opapi.so` 的目录；离线 ATC 场景则应指向包含 `op_proto`、`op_impl`、`op_tiling` 等交付件的 vendor 根目录。

**实践题 23｜自定义融合 Pass 阶段改动实验（对应 3.6）**

把 Pass 的注册阶段从 `kBeforeInferShape` 改为 `kAfterInferShape`，重新编译安装并 dump 图。

验收要点：说明两种阶段下 dump 图文件名的差异（`RunCustomPassBeforeInfershape` vs `RunCustomPass_AfterInferShape`），以及 InferShape 后阶段时 replacement 需要自行保证输出 shape 正确（必要时自行调用 shape 推导）。

**实践题 24｜Parser 导入改动实验（对应 3.2）**

准备一个 ONNX 模型，用 Parser/ATC 导入。尝试制造一个 GE 不识别的算子（自定义 op type），观察 ONNX Parser 是否报「type is not supported」。

验收要点：说明为什么会报错（ONNX Parser 不识别的 op type 直接失败），以及解决思路——编写 ONNX 解析插件（`REGISTER_CUSTOM_OP` + `ParseParamsFn`，op type 格式 `domain::version::OpType`）并提供 GE 侧自定义算子交付件。

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.07_answer.txt